# Phân loại cử chỉ tay HaGRID trên Kaggle

Notebook chạy lại các module hiện có trong dự án: chuẩn bị ảnh HaGRID, chia train/validation/test không trùng người dùng, huấn luyện mô hình và đánh giá trên tập test.

## Trước khi chạy

1. Trong Kaggle, chọn **Add Input** và thêm dataset `hagrid-sample-30k-384p`.
2. Đưa mã nguồn dự án vào `/kaggle/working/hand_gesture_classification` (ví dụ upload/clone project hoặc thêm project làm Kaggle Dataset rồi chép vào thư mục này). Nếu dùng đường dẫn khác, sửa `PROJECT_DIR` ở cell cấu hình.
3. Chọn GPU trong **Settings → Accelerator**. Bật Internet nếu cần tải trọng số pretrained ImageNet lần đầu.
4. Chạy các cell từ trên xuống. Thư mục `/kaggle/working` dùng để lưu dữ liệu đã xử lý và kết quả.

In [ ]:
from pathlib import Path

# Sửa các đường dẫn này nếu cấu trúc Kaggle Input của bạn khác.
PROJECT_DIR = Path("/kaggle/working/hand_gesture_classification")
ANN_DIR = Path("/kaggle/input/hagrid-sample-30k-384p/ann_train_val")
IMG_DIR = Path("/kaggle/input/hagrid-sample-30k-384p/hagrid_30k")
DATA_DIR = Path("/kaggle/working/data/processed")
OUTPUT_DIR = Path("/kaggle/working/outputs")

# Có thể giảm max_per_class để chạy thử nhanh hơn.
MAX_PER_CLASS = 2000
MODEL = "mobilenet_v2"  # cnn, mobilenet_v2, efficientnet_b0, resnet50
STRATEGY = "finetune"  # transfer models: feature_extract hoặc finetune; CNN: scratch
EPOCHS = 30
BATCH_SIZE = 64
NUM_WORKERS = 2

print(f"Project: {PROJECT_DIR}")
print(f"HaGRID annotations: {ANN_DIR}")
print(f"HaGRID images: {IMG_DIR}")
print(f"Processed data: {DATA_DIR}")
print(f"Outputs: {OUTPUT_DIR}")

In [ ]:
import os
import sys
import importlib.util

if not PROJECT_DIR.is_dir():
    raise FileNotFoundError(
        f"Không tìm thấy mã nguồn tại {PROJECT_DIR}. Hãy đưa project vào Kaggle working "
        "hoặc sửa PROJECT_DIR trong cell cấu hình."
    )
for name, path in (("ANN_DIR", ANN_DIR), ("IMG_DIR", IMG_DIR)):
    if not path.is_dir():
        raise FileNotFoundError(f"Không tìm thấy {name}: {path}. Hãy kiểm tra Add Input và đường dẫn.")

os.chdir(PROJECT_DIR)
if str(PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR))

required_modules = ["torch", "torchvision", "pandas", "PIL", "sklearn", "skimage", "tqdm"]
missing_modules = [name for name in required_modules if importlib.util.find_spec(name) is None]
if missing_modules:
    raise ModuleNotFoundError(
        f"Thiếu module: {', '.join(missing_modules)}. Chỉ cài các gói còn thiếu trong một cell riêng, "
        "ví dụ: %pip install <tên-gói>."
    )

import torch
from src.config import CLASSES

print("Python:", sys.version.split()[0])
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("Cảnh báo: đang chạy CPU; huấn luyện có thể chậm. Hãy bật GPU trong Kaggle Settings nếu cần.")
print("Các lớp:", CLASSES)

## 1. Chuẩn bị ảnh HaGRID

Script đọc annotation theo lớp, cắt vùng bàn tay và ghi ảnh 224×224 cùng `metadata.csv` vào `DATA_DIR`. `MAX_PER_CLASS` giới hạn số ảnh trên mỗi lớp.

In [ ]:
import subprocess

DATA_DIR.mkdir(parents=True, exist_ok=True)
prepare_command = [
    sys.executable, "-m", "src.data.prepare_hagrid",
    "--ann_dir", str(ANN_DIR),
    "--img_dir", str(IMG_DIR),
    "--out_dir", str(DATA_DIR),
    "--max_per_class", str(MAX_PER_CLASS),
]
print("Đang chuẩn bị ảnh HaGRID...")
subprocess.run(prepare_command, cwd=PROJECT_DIR, check=True)

## 2. Chia tập dữ liệu

Lệnh chia tập tạo `splits.csv` với tỷ lệ xấp xỉ 70/15/15 và nhóm theo `user_id`, giúp cùng một người không xuất hiện ở nhiều tập.

In [ ]:
split_command = [sys.executable, "-m", "src.data.split", "--data_dir", str(DATA_DIR)]
subprocess.run(split_command, cwd=PROJECT_DIR, check=True)

import pandas as pd

splits = pd.read_csv(DATA_DIR / "splits.csv")
display(splits["split"].value_counts().rename_axis("split").to_frame("images"))
display(pd.crosstab(splits["label"], splits["split"], margins=True))

users_by_split = {name: set(splits.loc[splits["split"] == name, "user_id"]) for name in ("train", "val", "test")}
assert not users_by_split["train"] & users_by_split["val"], "Phát hiện user_id trùng giữa train và val"
assert not users_by_split["train"] & users_by_split["test"], "Phát hiện user_id trùng giữa train và test"
assert not users_by_split["val"] & users_by_split["test"], "Phát hiện user_id trùng giữa val và test"
print("Đã xác nhận không có user_id bị trùng giữa các tập.")

## 3. Huấn luyện mô hình

Mặc định dùng MobileNetV2 fine-tune pretrained ImageNet. Có thể đổi `MODEL` và `STRATEGY` ở cell cấu hình; `cnn` chỉ dùng `scratch`. Model fine-tune sẽ huấn luyện head trong 5 epoch đầu rồi mở băng các block cuối. Kết quả được lưu trong thư mục con của `OUTPUT_DIR`.

In [ ]:
from src.config import CLASSICAL_MODELS, DL_MODELS

if MODEL not in DL_MODELS:
    raise ValueError(f"MODEL phải là một trong {DL_MODELS}, nhận được: {MODEL}")
if MODEL == "cnn" and STRATEGY != "scratch":
    raise ValueError("CNN tự thiết kế chỉ hỗ trợ STRATEGY='scratch'.")
if MODEL != "cnn" and STRATEGY not in ("feature_extract", "finetune"):
    raise ValueError("Mô hình transfer yêu cầu STRATEGY='feature_extract' hoặc 'finetune'.")

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
EXPERIMENT_NAME = f"{MODEL}_{STRATEGY}"
TRAIN_COMMAND = [
    sys.executable, "-m", "src.train_dl",
    "--model", MODEL,
    "--strategy", STRATEGY,
    "--data_dir", str(DATA_DIR),
    "--out_dir", str(OUTPUT_DIR),
    "--exp_name", EXPERIMENT_NAME,
    "--epochs", str(EPOCHS),
    "--batch_size", str(BATCH_SIZE),
    "--num_workers", str(NUM_WORKERS),
]
print("Thực thi:", " ".join(TRAIN_COMMAND))
subprocess.run(TRAIN_COMMAND, cwd=PROJECT_DIR, check=True)

EXPERIMENT_DIR = OUTPUT_DIR / EXPERIMENT_NAME
CHECKPOINT = EXPERIMENT_DIR / "best.pt"
if not CHECKPOINT.is_file():
    raise FileNotFoundError(f"Huấn luyện kết thúc nhưng không tìm thấy checkpoint: {CHECKPOINT}")
print("Checkpoint:", CHECKPOINT)

## 4. Đánh giá trên tập test

Tính accuracy, precision, recall, F1 macro và lưu ma trận nhầm lẫn cùng `eval.json` cạnh checkpoint.

In [ ]:
evaluate_command = [
    sys.executable, "-m", "src.evaluate",
    "--ckpt", str(CHECKPOINT),
    "--data_dir", str(DATA_DIR),
    "--batch_size", str(BATCH_SIZE),
    "--num_workers", str(NUM_WORKERS),
]
subprocess.run(evaluate_command, cwd=PROJECT_DIR, check=True)
print("Kết quả đánh giá:", EXPERIMENT_DIR / "eval.json")
print("Ma trận nhầm lẫn:", EXPERIMENT_DIR / "cm_test.png")

## Tùy chọn: so sánh thêm mô hình HOG

Cell này chạy Grid Search + cross-validation cho SVM, Random Forest và KNN; có thể mất nhiều thời gian. Đổi `RUN_CLASSICAL` thành `True` nếu muốn chạy. Sau đó tổng hợp những thí nghiệm đã có bằng `src.compare_results`.

In [ ]:
RUN_CLASSICAL = False

if RUN_CLASSICAL:
    classical_command = [
        sys.executable, "-m", "src.train_classical",
        "--model", *CLASSICAL_MODELS,
        "--data_dir", str(DATA_DIR),
        "--out_dir", str(OUTPUT_DIR),
    ]
    subprocess.run(classical_command, cwd=PROJECT_DIR, check=True)

compare_command = [sys.executable, "-m", "src.compare_results", "--out_dir", str(OUTPUT_DIR)]
subprocess.run(compare_command, cwd=PROJECT_DIR, check=True)

## Lưu kết quả trên Kaggle

Các file huấn luyện nằm trong `/kaggle/working/outputs` và dữ liệu đã xử lý trong `/kaggle/working/data/processed`. Chọn **Save Version → Save & Run All** để lưu phiên chạy và tải kết quả xuống từ phần Output của Kaggle.